# Getting Started with molfp**molfp** is a Python library for computing, interpreting, and analyzing molecular fingerprints. It provides 30 fingerprint types (13 2D + 9 3D + 8 graph-based) built on native RDKit, plus tools for explainability, SHAP interpretation, visualization, collision detection, activity cliff analysis, and more.```pip install molfp```

## 1. Computing Fingerprints

### 1.1 Single MoleculePass a SMILES string (or RDKit Mol), get back a list of integers.

In [ ]:
import molfpimport numpy as npaspirin = "CC(=O)Oc1ccccc1C(=O)O"fp = molfp.maccs(aspirin)print(f"MACCS:   {len(fp)} bits, {sum(fp)} ON")fp = molfp.ecfp4(aspirin)print(f"ECFP4:   {len(fp)} bits, {sum(fp)} ON")fp = molfp.pubchem(aspirin)print(f"PubChem: {len(fp)} bits, {sum(fp)} ON")

### 1.2 All 30 Fingerprint Types

In [ ]:
for name in molfp.available():    fn, n_bits, desc = molfp.get(name)    print(f"  {name:>30s}  {n_bits:>5d} dims  ({desc})")

### 1.3 Versatile InputEvery function accepts a single SMILES, list, pandas Series, or DataFrame.

In [ ]:
smiles = ["CCO", "c1ccccc1", aspirin, "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"]# List input -> numpy arrayfps = molfp.ecfp4(smiles)print(f"List input:  shape={fps.shape}")# Sparse + parallelfps_sp = molfp.klekota_roth(smiles, sparse=True, n_jobs=-1)print(f"Sparse KR:   shape={fps_sp.shape}, density={fps_sp.nnz/(fps_sp.shape[0]*fps_sp.shape[1]):.2%}")# compute_batch for any FP by namefps = molfp.compute_batch(smiles, "graph_standard")print(f"Graph std:   shape={fps.shape}")

## 2. Molecule Visualization

In [ ]:
# 2D drawingimg = molfp.draw(smiles, legends=["Ethanol", "Benzene", "Aspirin", "Caffeine"])img

In [ ]:
# 2D with fingerprint bit highlightingimg = molfp.draw(aspirin, highlight_bits=[88, 143, 162], fp_type="maccs")img

In [ ]:
# 3D interactive viewer (renders in Jupyter/Colab/Streamlit)molfp.view3d("CN1C=NC2=C1C(=O)N(C(=O)N2C)C", style="stick")

## 3. Structure StandardizationDesalt, neutralize, canonicalize tautomers, and deduplicate by InChIKey.

In [ ]:
cases = [    ("CC(=O)Oc1ccccc1C(=O)[O-].[Na+]", "sodium aspirin"),    ("O=C1CC=CC=C1",                    "cyclohexadienone"),]for smi, desc in cases:    print(f"  {desc:>20s}: {smi:>40s}  ->  {molfp.standardize_smiles(smi)}")# Full pipelineraw = ["CC(=O)Oc1ccccc1C(=O)O", "CC(=O)Oc1ccccc1C(=O)[O-].[Na+]", "INVALID", "CCO", "OCC"]result = molfp.preprocess(raw)print(f"\nInput: {result['n_input']} -> Standardized: {result['n_after_standardization']} -> Unique: {result['n_after_deduplication']}")

## 4. Bit-Level Explainability & Annotation

In [ ]:
# Which atoms set which bits?exp = molfp.explain(aspirin, "ecfp4")print(f"ECFP4: {len(exp)} ON bits explained")for bit, envs in list(exp.items())[:3]:    print(f"  bit {bit}: {envs}")# Chemical names for bitsprint()for bit in [83, 84, 274]:    ann = molfp.annotate("substructure", bit)    print(f"  bit {bit}: {ann['name']:>25s}  SMARTS: {ann.get('smarts','')[:35]}")

## 5. Distance & Similarity5 vectorized metrics: Tanimoto, Dice, Cosine, Sokal-Michener, Russel-Rao.

In [ ]:
fp_a = np.array(molfp.maccs("CCO"))fp_b = np.array(molfp.maccs(aspirin))print(f"Tanimoto: {molfp.tanimoto(fp_a, fp_b):.3f}")print(f"Dice:     {molfp.dice(fp_a, fp_b):.3f}")# Pairwise matrixfps = np.array(molfp.maccs(smiles))mat = molfp.pairwise(fps, "tanimoto")print(f"Pairwise: {mat.shape}")

## 6. Hash Collision Detection

In [ ]:
for size in [64, 256, 1024, 4096]:    coll = molfp.detect_collisions("c1ccc2c(c1)cc1ccc3ccccc3c1c2", "ecfp4", size)    n_on = sum(molfp.ecfp4("c1ccc2c(c1)cc1ccc3ccccc3c1c2", n_bits=size))    print(f"  {size:>5d} bits: {len(coll):>2d} collisions / {n_on:>3d} ON bits")

## 7. Bioactivity Profiling

In [ ]:
# Sample datasmi = ["CCO", "c1ccccc1", aspirin, "OC(=O)c1ccccc1O", "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",       "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O", "Nc1ccc(O)cc1", "c1ccncc1",       "c1ccc2ncccc2c1", "O=Cc1ccc(O)c(OC)c1"]act = [4.5, 5.0, 7.2, 6.8, 5.5, 7.0, 4.2, 5.8, 6.5, 4.8]labels = molfp.classify(act, metric="pchembl")print(f"Classes: {dict((l, labels.count(l)) for l in set(labels))}")enriched = molfp.enriched_bits(smi, labels, "maccs", min_enrichment=1.5)for cls, bits in sorted(enriched.items()):    print(f"  {cls}: {len(bits)} enriched bits")

## 8. Activity Cliffs

In [ ]:
cliffs = molfp.activity_cliffs(smi, act, "maccs", sim_threshold=0.3, activity_threshold=1.0)print(f"{len(cliffs)} cliffs found")for c in cliffs[:3]:    print(f"  sim={c['similarity']:.2f}  delta={c['delta_activity']:.1f}")

## 9. Consensus / Fusion Fingerprints

In [ ]:
result = molfp.fuse(smi, act, fp_types=["maccs", "estate", "pubchem"], n_bits=20, method="variance")print(f"Fused FP shape: {result['fused_fps'].shape}")sources = {}for col, (fp_type, bit) in result["source_map"].items():    sources[fp_type] = sources.get(fp_type, 0) + 1print(f"Bits per FP type: {sources}")

## 10. SHAP Interpretation with Chemical MeaningTrain a model, compute SHAP values, and map each important bit to its chemical meaning.

In [ ]:
result = molfp.interpret(smi, act, fp_type="maccs", metric="pchembl")print(f"Model accuracy: {result['accuracy']:.1%}")print("\nTop 5 important bits:")for item in result["feature_importance"][:5]:    print(f"  bit {item['bit']:>3d}  SHAP={item['mean_abs_shap']:.4f}  {item['chemical_meaning']}")

In [ ]:
# Per-class driversfor cls in result["label_set"]:    drivers = result["class_drivers"][cls]    pro = [d for d in drivers if d["direction"] == "promotes"][:2]    print(f"\n{cls.upper()}:")    for d in pro:        print(f"  ↑ {d['value_effect']:>7s}  {d['chemical_meaning']:>25s}  SHAP {d['mean_shap']:+.4f}")

## 11. Feature SelectionStandalone functions that work with any fingerprint output.

In [ ]:
# Compute a high-dimensional fingerprintX = np.array(molfp.klekota_roth(smi))y = [1 if l == "active" else 0 for l in labels]print(f"Original: {X.shape}")# Remove low-variance featuresresult = molfp.remove_low_variance(X, threshold=0.01)print(f"After variance filter: {result['n_remaining']} features (removed {result['n_removed']})")# Select top-k by mutual informationresult = molfp.select_features(X, y, k=50, threshold=0.01)print(f"After select_features: {result['X'].shape}")print(f"Top MI scores: {result['scores'][:5].round(3)}")

## 12. Molecular Graph RepresentationsConvert molecules to graph structures for GNNs (PyTorch Geometric, DGL) or fixed-length vectors for sklearn.4 graph types × 2 versions (base + enriched) = 8 graph fingerprints.

In [ ]:
# Standard graph: atoms=nodes, bonds=edgesg = molfp.mol_graph("c1ccccc1")print(f"Standard graph:")print(f"  atom_features: {g['atom_features'].shape}")print(f"  edge_index:    {g['edge_index'].shape}")print(f"  edge_features: {g['edge_features'].shape}")

In [ ]:
# All 4 graph typesfor t in ["standard", "distance", "line", "fragment"]:    g = molfp.mol_graph(aspirin, type=t)    print(f"  {t:10s}: {g['num_atoms']:>3d} nodes, {g['num_edges']:>4d} edges, "          f"node_dim={g['atom_features'].shape[1]}, edge_dim={g['edge_features'].shape[1]}")

In [ ]:
# Enriched: adds Morgan hashes, functional groups, neighbor counts per atomg_base = molfp.mol_graph(aspirin, type="standard")g_enr = molfp.mol_graph(aspirin, type="standard", enriched=True)print(f"Base atom features:     {g_base['atom_features'].shape[1]} dims")print(f"Enriched atom features: {g_enr['atom_features'].shape[1]} dims")

In [ ]:
# Graph vectors for sklearn (fixed-length)X = molfp.graph_vectors(smi, type="standard", enriched=True)print(f"Enriched graph vectors: {X.shape}")# Use in sklearn pipelinefrom sklearn.pipeline import make_pipelinefrom sklearn.ensemble import RandomForestClassifierpipe = make_pipeline(    molfp.GraphTransformer(enriched=True, type="standard"),    RandomForestClassifier(n_estimators=50, random_state=42))from sklearn.model_selection import cross_val_scorescores = cross_val_score(pipe, smi, y, cv=3, scoring="accuracy")print(f"Graph enriched -> RF: {scores.mean():.3f} ± {scores.std():.3f}")

## 13. scikit-learn IntegrationAll 30 FP types as sklearn transformers — works with Pipeline, GridSearchCV, FeatureUnion.

In [ ]:
from sklearn.pipeline import make_pipeline, make_unionfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.model_selection import cross_val_score# Pipeline: FP -> classifierpipe = make_pipeline(molfp.ECFPTransformer(n_bits=1024), RandomForestClassifier(n_estimators=50, random_state=42))scores = cross_val_score(pipe, smi, y, cv=3, scoring="accuracy")print(f"ECFP4 -> RF: {scores.mean():.3f} ± {scores.std():.3f}")# Feature union: combine FP typesunion_pipe = make_pipeline(    make_union(molfp.ECFPTransformer(n_bits=512), molfp.MACCSTransformer(), molfp.EStateTransformer()),    RandomForestClassifier(n_estimators=50, random_state=42))scores = cross_val_score(union_pipe, smi, y, cv=3, scoring="accuracy")print(f"ECFP+MACCS+EState -> RF: {scores.mean():.3f} ± {scores.std():.3f}")

## Summary| Category | Count | Key Functions ||----------|-------|---------------|| **2D Fingerprints** | 13 | `maccs()`, `ecfp4()`, `pubchem()`, `klekota_roth()`, ... || **3D Fingerprints** | 9 | `usr()`, `usrcat()`, `whim()`, `getaway()`, ... || **Graph Fingerprints** | 8 | `mol_graph()`, `graph_vectors()`, `to_pyg()`, `to_dgl()` || **Visualization** | — | `draw()`, `view3d()` || **Standardize** | — | `preprocess()`, `standardize_smiles()` || **Explain** | — | `explain()`, `annotate()` || **Interpret** | — | `interpret()`, `interpret_molecule()`, `compare_molecules()` || **Feature Selection** | — | `select_features()`, `remove_low_variance()` || **Distance** | — | `tanimoto()`, `pairwise()` — 5 vectorized metrics || **Collisions** | — | `detect_collisions()`, `collision_rate()` || **Bioactivity** | — | `classify()`, `enriched_bits()` || **Cliffs** | — | `activity_cliffs()`, `cliff_summary()` || **Fusion** | — | `fuse()`, `complementarity()` || **sklearn** | — | 22 transformers + `GraphTransformer` |